Reshape the core publication dataset (`04_dataset`) into one file ready to load into a database.


In [1]:
import sys
from pathlib import Path

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "data_helpers").is_dir() and (candidate / "checklists").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not locate the biodiversity repository root.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json

import pandas as pd

from data_helpers import results_config
from data_helpers.prep import db_dataset_prep

## 1. Configuration

In [2]:
RESULTS_CONFIG = results_config.load_results_config()
DB_CONFIG = RESULTS_CONFIG["db_dataset_prep"]
SOURCE_DIR = PROJECT_ROOT / RESULTS_CONFIG["biodiversity_evidence_prep"]["output_directory"]
OUTPUT_DIR = PROJECT_ROOT / DB_CONFIG["output_directory"]
DATASET_PATH = SOURCE_DIR / "dataset.parquet"
ABSTRACTS_PATH = SOURCE_DIR / "dataset_abstracts.parquet"
MANIFEST_PATH = OUTPUT_DIR / "manifest.json"
DB_CONFIG

{'description': 'Database-ready variant of the core dataset: abstracts joined in, Not Applicable blanked, country names instead of ISO3, audit/derived-flag columns dropped, id re-issued 1..N.',
 'output_directory': 'notebooks/data_processing/outputs/06_dataset_for_db'}

## 2. Build the database dataset

Reads the verified `04_dataset` outputs (no corpus rebuild) and applies the transformations in `data_helpers/prep/db_dataset_prep.py`.

In [3]:
dataset = pd.read_parquet(DATASET_PATH)
abstracts = pd.read_parquet(ABSTRACTS_PATH)
db, audit = db_dataset_prep.build_db_dataset(dataset, abstracts)

assert len(db) == len(dataset) == len(abstracts)
assert db["UT"].equals(dataset["UT"])
assert db["id"].tolist() == list(range(1, len(db) + 1))
assert not set(db_dataset_prep.DROPPED_COLUMNS) & set(db.columns)
assert db["abstract"].notna().sum() == abstracts["abstract"].notna().sum()
print(f"{len(db):,} rows x {len(db.columns)} columns (was {len(dataset.columns)}).")
display(pd.Series(audit["blanked_cells"], name="blanked_cells").rename_axis("column").reset_index().style.format({"blanked_cells": "{:,}"}).hide(axis="index"))
print(f"Missing abstracts: {audit['n_missing_abstract']:,}")
print(f"Country tokens kept verbatim (not valid ISO3): {len(audit['unconverted_country_tokens']):,} distinct, {audit['unconverted_country_tokens']['n_mentions'].sum():,} mentions")
display(audit["unconverted_country_tokens"].head(15).style.format({"n_mentions": "{:,}"}).hide(axis="index"))

605,199 rows x 35 columns (was 40).


column,blanked_cells
pred_regions,390
pred_subregions,"6,120"
pred_countries,"228,419"
locales,"297,343"
locale_coordinates,"547,374"
realm,"23,635"
biome,"12,219"


Missing abstracts: 0
Country tokens kept verbatim (not valid ISO3): 136 distinct, 258,313 mentions


token,n_mentions
Not Applicable,"228,987"
Unclear,"27,158"
TAN,633
MAD,231
ROM,210
UNC,122
BUL,100
DEN,85
UNS,83
UNR,83


## 3. Write and verify

`dataset_for_db.parquet` keeps native list fields. `dataset_for_db.csv` stores list fields as JSON text (e.g. `["Americas", "Europe and Central Asia"]`) and blanks as empty fields.

In [4]:
written = db_dataset_prep.write_db_dataset(db, OUTPUT_DIR)
reloaded = pd.read_parquet(written[0])
assert reloaded["id"].equals(db["id"]) and reloaded["UT"].equals(db["UT"])
assert reloaded.columns.tolist() == db.columns.tolist()
assert reloaded["pred_countries"].isna().sum() == db["pred_countries"].isna().sum()
csv_head = pd.read_csv(written[1], nrows=1000, dtype=str, keep_default_na=False)
assert csv_head.columns.tolist() == db.columns.tolist()

manifest = {
    "source_dataset": str(DATASET_PATH.relative_to(PROJECT_ROOT)),
    "source_abstracts": str(ABSTRACTS_PATH.relative_to(PROJECT_ROOT)),
    "rows": len(db),
    "columns": db.columns.tolist(),
    "dropped_columns": list(db_dataset_prep.DROPPED_COLUMNS),
    "blanked_cells": audit["blanked_cells"],
    "n_missing_abstract": audit["n_missing_abstract"],
    "country_tokens_not_converted": {
        row.token: int(row.n_mentions) for row in audit["unconverted_country_tokens"].itertuples()
    },
    "id": "re-issued 1..N after all other steps; unrelated to the WOS UT suffix",
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
print("Verified database dataset:")
for path in [*written, MANIFEST_PATH]:
    print(f"  {path.relative_to(PROJECT_ROOT)} ({path.stat().st_size / 1024**2:.2f} MiB)")

Verified database dataset:
  notebooks/data_processing/outputs/06_dataset_for_db/dataset_for_db.parquet (627.92 MiB)
  notebooks/data_processing/outputs/06_dataset_for_db/dataset_for_db.csv (1406.72 MiB)
  notebooks/data_processing/outputs/06_dataset_for_db/manifest.json (0.00 MiB)


In [5]:
db.drop(columns="abstract").head(3).T

,0,1,2
id,1,2,3
UT,WOS:001410940900001,WOS:001498294300001,WOS:001501498700001
title,Skeletal deformities in Sparus aurata from Tür...,Size-dependent effects of oxo-degradable plast...,Traditional and biodegradable plastics improve...
publication_year,2026,2026,2026
doi,10.1111/azo.12538,10.15302/J-FASE-2025623,10.15302/J-FASE-2025626
s1_r,1,1,1
s2_r,-1,1,1
s3_r,-1,1,1
s4_r,1,1,1
s1_bio,"[""ecosystem_change""]","[""species_change"", ""community_change""]","[""community_change"", ""ecosystem_change""]"
